In [14]:
import torch 
import torch.nn as nn

In [15]:
cfg=GPT_2_config={
  "vocab":50257,
  "context_len":1024,
  "emb_dim":768,
  "n_head":12,
  "n_layer":12,
  "dropout":0.1,
  "qkv_bias":False  
}

In [ ]:
class  DummyNorm(nn.Module):
  def __init__(self,emb_dim):
    super().__init__()
  def forward(self,x):
    return x  

In [ ]:
class DummyTransformerBlock(nn.Module):
  def __init__(self,cfg):
    super().__init__()
  def forward(self,x):
    return x

In [ ]:
class DummyGPT(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    
    self.token_emb=nn.Embedding(cfg["vocab"],cfg["emb_dim"])
    self.pos_emb=nn.Embedding(cfg["context_len"],cfg["emb_dim"])
    self.dropout=nn.Dropout(cfg["dropout"])
    self.tranformer_block= nn.Sequential(
            *[DummyTransformerBlock(cfg) for _ in range(cfg["n_layer"])]
        )
    self.final_norm=DummyNorm(cfg["emb_dim"])
    self.out_head=nn.Linear(cfg["emb_dim"],cfg["vocab"])
    
    
  def forward(self,ip_batch):
      batch,seq_len=ip_batch.shape
      
      token_embd=self.token_emb(ip_batch)
      pos_embd=self.pos_emb(torch.arange(seq_len))
      
      x=token_embd+pos_embd
      
      x=self.dropout(x)
      
      x=self.tranformer_block(x)
      
      x=self.final_norm(x)
      
      logit=self.out_head(x)
      
      return logit
       

In [17]:
import tiktoken
tokenizer=tiktoken.get_encoding("gpt2")
batch=[]

In [18]:
text1="How are you go"
text2="How go are you"

batch.append(torch.tensor(tokenizer.encode(text1)))
batch.append(torch.tensor(tokenizer.encode(text2)))
batch = torch.stack(batch, dim=0)
print(batch.shape)

torch.Size([2, 4])


In [ ]:
torch.manual_seed(123)

model=DummyGPT(cfg)

logit=model(batch)

print(logit.shape)

torch.Size([2, 4, 50257])


In [25]:
token_ids = torch.argmax(logit[0], dim=-1)   # shape: (seq_len,)
text = tokenizer.decode(token_ids.tolist())   # convert to list first
print(text)

 surg illegally Franks Voyager
